# 02. Q-Network với đầu ra Dueling

Q-Network nhận 12 số của observation và trả về hai giá trị Q(s, IDLE) và Q(s, FLAP). Notebook này chỉ dùng Keras, không biết gì về trò chơi.

Nội dung:

1. Hàm `build_q_network`.
2. Kiểm tra nhanh.

Cấu hình lấy từ mục 5 của `Implementation_Notes.md`.

## 1. Kiến trúc

MLP gồm hai lớp ẩn 256 nơ-ron dùng ReLU, sau đó chia thành hai nhánh:

```
h      = ReLU(W₂ · ReLU(W₁ · s))
V(s)   = w_V · h                         phần chung cho mọi hành động
A(s,a) = W_A · h                         phần chênh lệch giữa các hành động
Q(s,a) = V(s) + A(s,a) − (trung bình theo a' của A(s,a'))
```

Phép trừ trung bình làm V và A được xác định duy nhất: nếu không có nó, cộng c vào V và trừ c khỏi A vẫn cho cùng một Q. Mạng có 69.891 tham số.

Khi lưu checkpoint, dự án chỉ lưu trọng số (`.weights.h5`), rồi dựng lại mạng bằng hàm này khi tải.

In [ ]:
import numpy as np
import keras


def build_q_network(obs_dim=12, n_actions=2, hidden_units=256):
    states = keras.Input(shape=(obs_dim,), name="state")
    h = keras.layers.Dense(hidden_units, activation="relu", name="hidden_1")(states)
    h = keras.layers.Dense(hidden_units, activation="relu", name="hidden_2")(h)
    value = keras.layers.Dense(1, name="value")(h)
    advantage = keras.layers.Dense(n_actions, name="advantage")(h)
    q_values = value + advantage - keras.ops.mean(advantage, axis=1, keepdims=True)
    return keras.Model(states, q_values, name="dueling_q_network")

## 2. Kiểm tra nhanh

- Đầu ra có dạng (batch, 2) và số tham số đúng 69.891.
- Trung bình Q theo hành động bằng đúng V(s), là tính chất của phép trừ trung bình ở trên.

In [ ]:
def _self_check_q_network():
    net = build_q_network()
    assert net.count_params() == 69_891, net.count_params()

    states = np.random.default_rng(0).uniform(-1, 1, size=(5, 12)).astype(np.float32)
    q = net(states).numpy()
    assert q.shape == (5, 2)

    value_net = keras.Model(net.input, net.get_layer("value").output)
    v = value_net(states).numpy()[:, 0]
    assert np.allclose(q.mean(axis=1), v, atol=1e-5)
    print("02_q_network: kiểm tra đạt.")


_self_check_q_network()